# xGoal Model Prototyping

Interactive port of `nn_xgoals.py` for rapid local prototyping. Edit the parameters cell below to change the training-season window and the held-out season to evaluate on, then run all cells.

Requires shot CSVs in `data/` (see `download_data.sh` at the repo root).

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.calibration import calibration_curve
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, auc

from scipy.stats import describe, linregress

import torch
import torch.nn as nn
import torch.nn.functional as F

%matplotlib inline

## Parameters

Edit these to prototype quickly — no need to touch anything below.

In [ ]:
TRAIN_START_YEAR = 2021  # first training season (2021 -> 2021-2022)
TRAIN_END_YEAR = 2024    # last training season, inclusive (2024 -> 2024-2025)
PREDICT_YEAR = 2025      # held-out season to evaluate xG predictions on (2025 -> 2025-2026)

shots_dir = 'data/'    # relative to repo root; populate with ../download_data.sh
plot_folder = 'plots/'  # kept separate from cluster_plots/ so this notebook doesn't overwrite the cluster run's reference plots
os.makedirs(plot_folder, exist_ok=True)

fn_shots = [os.path.join(shots_dir, f'shots_{y}_{y + 1}.csv') for y in range(TRAIN_START_YEAR, TRAIN_END_YEAR + 1)]
fn_predict = os.path.join(shots_dir, f'shots_{PREDICT_YEAR}_{PREDICT_YEAR + 1}.csv')

# Apple Silicon GPU acceleration if available, falls back to CPU
device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')

print(f'Training seasons: {TRAIN_START_YEAR}-{TRAIN_START_YEAR + 1} through {TRAIN_END_YEAR}-{TRAIN_END_YEAR + 1} ({len(fn_shots)} seasons)')
print(f'Prediction season: {PREDICT_YEAR}-{PREDICT_YEAR + 1}')
print(f'Device: {device}')

## Load Data

In [ ]:
# read each training season CSV and concatenate
data = pd.concat([pd.read_csv(file) for file in fn_shots], ignore_index=True)
data = data[data['isPlayoffGame'] == 0]
data = data[~data['lastEventCategory'].isin(['PENL', 'STOP', 'GOAL', 'CHL', 'PEND', 'PSTR', 'ANTHEM', 'EISTR', 'GEND', 'EGT'])]

print(f'Training shots: {len(data):,}')
data.head()

In [ ]:
# held-out season used for final evaluation
data_predict = pd.read_csv(fn_predict)
data_predict = data_predict[data_predict['isPlayoffGame'] == 0]
data_predict = data_predict[~data_predict['lastEventCategory'].isin(['PENL', 'STOP', 'GOAL', 'CHL', 'PEND', 'PSTR', 'ANTHEM', 'EISTR', 'GEND', 'EGT'])]

print(f'Prediction shots: {len(data_predict):,}')
data_predict.head()

## Feature Selection

In [ ]:
y = data['goal']

features = ['timeSinceLastEvent', # Time between the shot and the event that took place before the shot
            'shotAngle', # The angle of the shot in degrees. Is a positive number if the shot is from the left side of the ice.
            'shotAnglePlusRebound', # The angle of the shot in degrees. Is a positive number if the shot is from the left side of the ice.
            'shotAngleReboundRoyalRoad', # Set to 1 if the puck went through the middle of the between this shot and previous shot if this shot is a rebound.
            'shotDistance', # The distance from the net of the shot in feet. Net is defined as being at the (89,0) coordinates
            'shotType', # Type of the shot. (Slap, Wrist, etc)
            'shotRebound', # Set to 1 if the shot is a rebound. (If the last event was a shot and within 3 seconds of this shot)
            'shotAnglePlusReboundSpeed', # The shotAnglePlusRebound variable divided by time between the last shot and this one. (How fast the angle changed)
            'speedFromLastEvent', # The distance between the shot location and the previous event's location divided by the number of seconds between them
            'distanceFromLastEvent', # The distance between the shot location and the previous event's location in feet
            'lastEventShotAngle', # The shot angle of the shot directly before this shot. (If the last event was a shot)
            'lastEventShotDistance', # The shot distance of the shot directly before this shot. (If the last event was a shot)
            'lastEventCategory', # The type of event before the shot. Shot, hit, etc.
            'shooterTimeOnIce', # playing time in seconds that have passed since the shooter started their shift
            'shootingTeamAverageTimeOnIce', # The average playing time in seconds the shooting team's players have been on the ice
            'defendingTeamAverageTimeOnIce', # The average playing time in seconds the defending team's players have been on the ice
            'offWing', # Set to 1 if the shot is from the left side of the ice and the shooter is a right shot, or vice-versa. Otherwise 0
            'shotOnEmptyNet'] # set to 0 if the net is empty, 1 if non-empty

print('Number of features:', len(features))

X = data[features]
X_predict = data_predict[features]

p_num = X.select_dtypes(include=['int64', 'float64'])
p_cat = X.select_dtypes(include=['object'])

## Preprocessing

In [ ]:
preprocessor = ColumnTransformer([
    ('cat_encoder', OneHotEncoder(handle_unknown='ignore'), p_cat.columns)
], remainder='passthrough')

X_postprocess = preprocessor.fit_transform(X)
X_predict_postprocess = preprocessor.fit_transform(X_predict)

X_postprocess_df = pd.DataFrame(X_postprocess, columns=preprocessor.get_feature_names_out())
X_predict_postprocess_df = pd.DataFrame(X_predict_postprocess, columns=preprocessor.get_feature_names_out())

In [ ]:
corr_threshold = 0.9
corr_matrix = X_postprocess_df.corr().abs()
upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper_tri.columns if any(upper_tri[col] > corr_threshold)]

X_postprocess_df = X_postprocess_df.drop(columns=to_drop)
X_predict_postprocess_df = X_predict_postprocess_df.drop(columns=to_drop)
print('Correlated features dropped:', to_drop)

In [ ]:
var_thresh = VarianceThreshold(threshold=0.005)
X_postprocess_filtered = var_thresh.fit_transform(X_postprocess_df)
selected_columns = X_postprocess_df.columns[var_thresh.get_support()]

X_postprocess_filtered = pd.DataFrame(X_postprocess_filtered, columns=selected_columns)
X_predict_postprocess_df = pd.DataFrame(X_predict_postprocess_df, columns=selected_columns)

print(f'Features after pruning: {X_postprocess_filtered.shape[1]}')

## Train / Val Split

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(
    X_postprocess_filtered.to_numpy(),
    y,
    test_size=0.2,
    random_state=1997,
    stratify=y,
    shuffle=True,
)
# stratify=y ensures goal/no-goal balance matches between train and val sets

X_train_tensor = torch.FloatTensor(X_train).to(device)
y_train_tensor = torch.LongTensor(y_train).to(device)

y_val = torch.LongTensor(y_val.to_numpy()).to(device)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = y_val.unsqueeze(1).float().to(device)

## Model Definition

In [ ]:
class XGoalNeuralNetwork(nn.Module):

    def __init__(self, input_dim, h1=512, h2=256, h3=128, h4=64, out_features=1):
        super().__init__()

        self.fc1 = nn.Linear(input_dim, h1)
        self.bn1 = nn.BatchNorm1d(h1)
        self.fc2 = nn.Linear(h1, h2)
        self.bn2 = nn.BatchNorm1d(h2)
        self.fc3 = nn.Linear(h2, h3)
        self.bn3 = nn.BatchNorm1d(h3)
        self.fc4 = nn.Linear(h3, h4)
        self.out = nn.Linear(h4, out_features)

        self.dropout = nn.Dropout(p=0.3)

    def forward(self, x):
        x = F.leaky_relu(self.bn1(self.fc1(x)), negative_slope=0.01)
        x = self.dropout(x)

        x = F.leaky_relu(self.bn2(self.fc2(x)), negative_slope=0.01)
        x = self.dropout(x)

        x = F.leaky_relu(self.bn3(self.fc3(x)), negative_slope=0.01)
        x = self.dropout(x)

        x = F.relu(self.fc4(x))
        x = self.out(x)
        return x

## Train the Model

In [ ]:
epochs = 1000  # early stopping will typically end training well before this

torch.manual_seed(1997)
input_dim = X_train_tensor.shape[1]
model = XGoalNeuralNetwork(input_dim).to(device)

target_lr = 1e-3
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=target_lr, weight_decay=1e-4)

warmup_epochs = 20
warmup_start_lr = 1e-6
for param_group in optimizer.param_groups:
    param_group['lr'] = warmup_start_lr

scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=40, gamma=0.1)

patience = 30
best_val_loss = float('inf')
counter = 0

losses = []
losses_val = []

for i in range(epochs):
    model.train()
    optimizer.zero_grad()

    y_train_calc = model(X_train_tensor)
    loss_train = criterion(y_train_calc, y_train_tensor.unsqueeze(1).float())
    losses.append(loss_train.item())

    loss_train.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        y_val_calc = model(X_val_tensor)
        loss_test = criterion(y_val_calc, y_val_tensor)

    losses_val.append(loss_test.item())

    if i % 10 == 0:
        current_lr = optimizer.param_groups[0]['lr']
        print(f"Epoch {i:4d} | Training Loss: {loss_train.item():.4f} | Validation Loss: {loss_test.item():.4f} | log LR: {np.log10(current_lr):.4f}")

    if loss_test.item() < best_val_loss:
        best_val_loss = loss_test.item()
        counter = 0
        best_model_state = model.state_dict()
    else:
        counter += 1

    if counter >= patience:
        print(f'Early stopping activated at epoch {i}')
        epochs = i
        break

    if i < warmup_epochs:
        warmup_lr = warmup_start_lr + (target_lr - warmup_start_lr) * (i / warmup_epochs)
        for param_group in optimizer.param_groups:
            param_group['lr'] = warmup_lr
    else:
        scheduler.step()

model.load_state_dict(best_model_state)

## Training Diagnostics

In [ ]:
fig, (ax1, ax2) = plt.subplots(
    2, 1, figsize=(8, 6),
    gridspec_kw={'height_ratios': [3, 1]},
    sharex=True,
)

ax1.plot(range(len(losses)), losses, label='Training Loss')
ax1.plot(range(len(losses_val)), losses_val, label='Validation Loss')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss (with regularization)')
ax1.legend()
ax1.grid(True)

residuals = [train / val if val != 0 else float('nan') for train, val in zip(losses, losses_val)]
ax2.plot(range(len(losses)), residuals, color='purple')
ax2.set_xlabel('Epochs')
ax2.set_ylabel('Residual')
ax2.set_title('Training / Validation Loss')
ax2.grid(True)

final_residual = residuals[-1]
textstr = f'Final Residual: {final_residual:.5f}'
props = dict(boxstyle='round', facecolor='white', alpha=0.8)
ax2.text(0.95, 0.1, textstr, transform=ax2.transAxes, fontsize=10, bbox=props, ha='right', va='bottom')

plt.tight_layout()
fig.savefig(plot_folder + 'losses.png')
plt.show()

In [ ]:
train_fpr, train_tpr, _ = roc_curve(y_train_tensor.cpu().numpy(), y_train_calc.detach().cpu().numpy())
val_fpr, val_tpr, _ = roc_curve(y_val.cpu().numpy(), y_val_calc.detach().cpu().numpy())

plt.figure()
plt.plot(train_fpr, train_tpr, label=f'Train AUC = {roc_auc_score(y_train_tensor.cpu().numpy(), y_train_calc.detach().cpu().numpy()):.2f}')
plt.plot(val_fpr, val_tpr, label=f'Val AUC = {roc_auc_score(y_val.cpu().numpy(), y_val_calc.detach().cpu().numpy()):.2f}')
plt.plot([0, 1], [0, 1], 'k--', label='Random')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.savefig(plot_folder + 'roc_curve.png')
plt.show()

In [ ]:
precision, recall, thresholds = precision_recall_curve(y_val.cpu().numpy(), y_val_calc.detach().cpu().numpy())
pr_auc = auc(recall, precision)

plt.figure()
plt.plot(recall, precision, label=f'PR AUC = {pr_auc:.2f}')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend()
plt.savefig(plot_folder + 'pr_curve.png')
plt.show()

In [ ]:
true = y_val.cpu().numpy()
pred_probs = torch.sigmoid(y_val_calc).detach().cpu().numpy()

prob_true, prob_pred = calibration_curve(true, pred_probs, n_bins=30)

plt.figure()
plt.plot(prob_pred, prob_true, marker='o', label='Model')
plt.plot([0, 1], [0, 1], linestyle='--', label='Perfect Calibration')
plt.xlabel('Predicted Probability')
plt.ylabel('Observed Frequency')
plt.title('Calibration Curve')
plt.legend()
plt.savefig(plot_folder + 'calibration_curve.png')
plt.show()

## Predict on Held-Out Season

In [ ]:
model.eval()
with torch.no_grad():
    y_pred_logits = model(torch.tensor(X_predict_postprocess_df.to_numpy(), dtype=torch.float32).to(device))
    T = 1.0  # temperature to scale sigmoid
    y_pred = torch.sigmoid(y_pred_logits / T).cpu()

number_of_bins = int(np.sqrt(np.shape(data)[0]) / 8)

plt.figure(dpi=300)
plt.hist(y_pred.numpy(), bins=number_of_bins)
plt.xlabel('Probability of goal')
plt.ylabel('Number of shots')
plt.savefig(plot_folder + 'xgoal_hist.png')
plt.show()

## Results Tables

In [ ]:
pd.set_option('display.max_rows', None)

df = data_predict
df['pred_xgoal'] = y_pred

grouped_pred = df.groupby('shooterName').agg({'xGoal': 'sum', 'goal': 'sum', 'pred_xgoal': 'sum'})
grouped_pred['goal_residual'] = grouped_pred['goal'] - grouped_pred['xGoal']
grouped_pred['pred_goal_residual'] = grouped_pred['goal'] - grouped_pred['pred_xgoal']
grouped_pred['residual_difference'] = grouped_pred['pred_goal_residual'] - grouped_pred['goal_residual']


def format_table(table):
    out = table.reset_index().rename(columns={
        'shooterName': 'Shooter',
        'pred_xgoal': 'xGoal (model)',
        'goal': 'Goals',
        'pred_goal_residual': 'Goals - xGoal',
    })[['Shooter', 'xGoal (model)', 'Goals', 'Goals - xGoal']]
    out['xGoal (model)'] = out['xGoal (model)'].round(3)
    out['Goals - xGoal'] = out['Goals - xGoal'].round(3)
    out['Goals'] = out['Goals'].astype(int)
    return out

In [ ]:
# players whose actual goals fell furthest short of model xG (biggest negative residual)
sorted_grouped_pred = grouped_pred.sort_values(by='pred_goal_residual', ascending=True)

print('Biggest underperformers vs. model xG:')
display(format_table(sorted_grouped_pred.head(20)))

In [ ]:
# players whose actual goals exceeded model xG by the most
print('Biggest overperformers vs. model xG:')
display(format_table(sorted_grouped_pred.tail(20)[::-1]))

In [ ]:
min_goal_for_chart = 25
filtered_shooters = sorted_grouped_pred[sorted_grouped_pred['goal'] >= min_goal_for_chart]
closest_to_zero = filtered_shooters.iloc[filtered_shooters['pred_goal_residual'].abs().argsort()[:30]]

print(f'Most accurate predicted players (min {min_goal_for_chart} goals):')
display(format_table(closest_to_zero))

In [ ]:
sorted_grouped_pred_actual_goals = grouped_pred.sort_values(by='pred_xgoal', ascending=False)

print('Top predicted goal scorers, by model xG:')
display(format_table(sorted_grouped_pred_actual_goals.head(30)))

## Actual vs. Predicted Regression

In [ ]:
x = grouped_pred['goal'].values
y_actual_vs_pred = grouped_pred['pred_xgoal'].values

slope, intercept, r_value, p_value, std_err = linregress(x, y_actual_vs_pred)

x_vals = np.linspace(x.min(), x.max(), 100)
y_line = slope * x_vals + intercept

resid = y_actual_vs_pred - (slope * x + intercept)
rse = np.sqrt(np.sum(resid ** 2) / (len(x) - 2))
x_mean = np.mean(x)
n = len(x)
se_line = rse * np.sqrt(1 / n + (x_vals - x_mean) ** 2 / np.sum((x - x_mean) ** 2))
ci = 1.96 * se_line
lower = y_line - ci
upper = y_line + ci

plt.figure(figsize=(10, 5), dpi=150)
sns.scatterplot(x=x, y=y_actual_vs_pred, color='k', alpha=0.7, label='Data')
sns.lineplot(x=x, y=x, color='cyan', linestyle='--', label='Ideal Predictions')
sns.lineplot(x=x_vals, y=y_line, color='blue', label='Regression Line')
plt.fill_between(x_vals, lower, upper, color='blue', alpha=0.2, label='95% CI')

eq_text = f"y = {slope:.2f}x + {intercept:.2f}\n$R^2$ = {r_value ** 2:.2f}"
plt.text(0.05, 0.85, eq_text, transform=plt.gca().transAxes, fontsize=12, bbox=dict(facecolor='white', alpha=0.5))

plt.xlabel('Actual Goals')
plt.ylabel('Predicted Goals')
plt.title(f'Actual vs. Predicted Goals: {PREDICT_YEAR}-{PREDICT_YEAR + 1}, with T = {T}')
plt.legend()
plt.savefig(plot_folder + 'regression_line_pred_vs_real.png')
plt.show()

## Distribution Comparisons

In [ ]:
bins__ = 50

plt.figure(dpi=100)
plt.title('Compare distributions of goals vs xgoals')
plt.hist(grouped_pred['goal'], bins=bins__, histtype='step', color='blue', label='goals')
plt.hist(grouped_pred['pred_xgoal'], bins=bins__, histtype='step', color='red', label='xgoals')
plt.xlabel('Goals scored')
plt.legend()
plt.savefig(plot_folder + 'compare_distributions.png')
plt.show()

print(describe(grouped_pred['goal']))
print(describe(grouped_pred['pred_xgoal']))

In [ ]:
plt.figure(dpi=100)
plt.title('Compare AG model vs MP model')
plt.hist(grouped_pred['goal_residual'], bins=bins__, histtype='step', color='blue', label='MP')
plt.hist(grouped_pred['pred_goal_residual'], bins=bins__, histtype='step', color='red', label='AG')
plt.xlabel('goals - xgoals')
plt.legend()
plt.savefig(plot_folder + 'ag_vs_mp.png')
plt.show()

print(describe(grouped_pred['goal_residual']))
print(describe(grouped_pred['pred_goal_residual']))